# 16-01 从候选比较到重新规划

本实验对应第 16 章，按三层顺序运行：先从三条现成动作中排除无效候选并选出较好结果；再使用 CEM 主动搜索候选；最后比较“一次计划全部执行”和“执行两步后重新观察”。依赖 NumPy 与 Matplotlib，CPU 即可。

实验使用明确给出的二维位置更新规则作为教学世界模型，不训练视觉模型，也不模拟真实接触。它只验证预测、目标评分、搜索和重新观察之间的数据关系。


## 1. 先比较三条已经给出的候选

起点为 `[0.10, 0.10]`，目标为 `[0.80, 0.60]`。候选 A 合法且接近目标；候选 B 合法但移动不足；候选 C 的预测终点最好，却在第一步超过单轴 0.12 的动作限制。

运行后请先看打印表，再看路径图。选择顺序必须是“先检查动作是否有效，再在有效候选中比较目标距离”。


In [ ]:
import numpy as np  # 计算候选轨迹、目标距离和搜索结果
import matplotlib.pyplot as plt  # 显示候选路径与重新规划效果
candidate_start = np.array([0.10, 0.10])  # 设置三条候选共同使用的当前状态
candidate_goal = np.array([0.80, 0.60])  # 设置本轮动作选择的任务目标
candidate_limit = 0.12  # 设置每个动作分量允许的最大绝对值
candidate_a = np.tile(np.array([0.080, 0.055]), (8, 1))  # 构造合法且接近目标的候选 A
candidate_b = np.tile(np.array([0.040, 0.020]), (8, 1))  # 构造合法但移动不足的候选 B
candidate_c = np.tile((candidate_goal - candidate_start) / 8, (8, 1))  # 先构造终点恰好到达目标的候选 C
candidate_c[0, 0] = 0.160  # 让候选 C 的第一步故意超过动作限制
candidate_c[1, 0] = 0.015  # 调整第二步以保持候选 C 的总位移不变
candidate_actions = [candidate_a, candidate_b, candidate_c]  # 按 A、B、C 收集三条完整动作块
candidate_names = ['A', 'B', 'C']  # 设置打印和绘图使用的候选名称
candidate_paths = [candidate_start + np.cumsum(actions, axis=0) for actions in candidate_actions]  # 用同一个教学世界模型预测三条路径
candidate_valid = [bool(np.all(np.abs(actions) <= candidate_limit)) for actions in candidate_actions]  # 检查每条动作块是否满足单步限制
candidate_scores = [float(np.linalg.norm(path[-1] - candidate_goal)) for path in candidate_paths]  # 计算每条预测终点到目标的距离
valid_ids = [index for index, valid in enumerate(candidate_valid) if valid]  # 找出可以参加目标比较的候选编号
selected_id = min(valid_ids, key=lambda index: candidate_scores[index])  # 只在合法候选中选目标距离最小者
for name, valid, score, path in zip(candidate_names, candidate_valid, candidate_scores, candidate_paths):  # 逐条打印预测和检查结果
    print(f'候选 {name}：预测终点 {path[-1].round(3)}，目标距离 {score:.3f}，动作有效={valid}')  # 显示选择所需的三项信息
print(f'最终选择候选 {candidate_names[selected_id]}，因为它是合法候选中预测结果最接近目标的一条。')  # 说明最终选择依据
plt.figure(figsize=(6, 4.5))  # 创建候选路径对照图
for name, valid, path in zip(candidate_names, candidate_valid, candidate_paths):  # 依次绘制三条预测轨迹
    style = '-' if valid else '--'  # 用虚线标出因动作越界而无效的候选
    plt.plot(np.r_[candidate_start[0], path[:, 0]], np.r_[candidate_start[1], path[:, 1]], marker='o', linestyle=style, label=f'{name}: valid={valid}')  # 显示候选从相同起点出发的预测路径
plt.scatter(*candidate_goal, marker='*', s=160, color='black', label='goal')  # 标出任务目标
plt.xlabel('x position')  # 标明横轴是二维位置的第一维
plt.ylabel('y position')  # 标明纵轴是二维位置的第二维
plt.title('Check constraints before comparing goal distance')  # 强调先检查约束再比较目标
plt.axis('equal')  # 使用相同比例避免路径方向失真
plt.grid(alpha=0.25)  # 添加浅色网格帮助读取位置
plt.legend(fontsize=8)  # 显示候选有效性与目标
plt.show()  # 输出三条候选的预测结果


### 为什么不能直接选择 C

C 的终点距离最小，但第一步动作超过允许范围，因此它在目标评分之前已经被排除。A 虽然没有精确到达星号，却是合法候选中最接近目标的一条。B 也合法，只是预测移动不足。

这张图把三个角色分开了：教学世界模型负责把动作累加成未来路径；动作限制负责判断能不能执行；目标距离负责在剩余路径中排序。任何一个分数都不能同时替代三项工作。


## 2. 现成候选不够时，用 CEM 继续搜索

接下来回到更长的二维移动任务。CEM 每轮采样 384 条十二步动作块，使用同一个预测器计算后果，保留 38 条较好候选，再调整下一轮的采样均值和标准差。

运行后得到的曲线不是训练损失。世界模型始终不变，下降的是当前搜索已找到的最低规划代价。


In [ ]:
import numpy as np  # 计算批量候选轨迹
import matplotlib.pyplot as plt  # 显示路径与误差
rng = np.random.default_rng(16)  # 固定搜索随机数以便复查结果
goal = np.array([0.45, 0.3])  # 指定二维目标位置
start = np.array([-0.45, -0.3])  # 指定两种控制方式的共同起点
limit = 0.12  # 限制每个动作坐标的最大位移
horizon = 12  # 每次预测未来十二步
def predict_paths(state, actions):  # 定义固定且已知的教学预测器
    return state + np.cumsum(actions, axis=1)  # 沿时间累加每条候选动作
def costs(state, actions):  # 定义所有候选共用的评价规则
    paths = predict_paths(state, actions)  # 先计算候选导致的位置变化
    distances = ((paths - goal) ** 2).sum(axis=2)  # 计算各步到目标的平方距离
    return distances[:, -1] + 0.15 * distances.mean(axis=1) + 0.02 * (actions ** 2).sum(axis=(1, 2))  # 同时评价终点、过程与控制幅度
def cem(state, steps=horizon):  # 在固定世界预测器上搜索动作而不训练网络
    mean = np.tile(np.clip((goal - state) / steps, -limit, limit), (steps, 1))  # 将直接接近目标作为可解释的初始化
    std = np.full_like(mean, 0.07)  # 给各步动作保留搜索空间
    best = mean.copy()  # 保存一个已经评分过的具体候选
    best_score = costs(state, best[None])[0]  # 计算初始化候选的真实规划代价
    history = []  # 保存每轮搜索后的最低已知代价
    for iteration in range(5):  # 固定每次搜索的迭代预算
        samples = np.clip(rng.normal(mean, std, size=(384, steps, 2)), -limit, limit)  # 采样并保证动作合法
        samples[0] = best  # 保留上一轮最好的候选避免遗忘
        scores = costs(state, samples)  # 使用同一个模型与评分规则比较候选
        elite_ids = np.argsort(scores)[:38]  # 选出本轮代价最低的一小部分
        elite = samples[elite_ids]  # 取得具体动作序列而不是平均预测图像
        mean = elite.mean(axis=0)  # 更新每个动作位置的采样中心
        std = np.maximum(elite.std(axis=0), 0.008)  # 保留标准差下限避免搜索过早停止
        if scores[elite_ids[0]] < best_score:  # 仅当找到更好的具体动作时替换保留结果
            best = elite[0].copy()  # 保存实际评估过的动作块
            best_score = scores[elite_ids[0]]  # 保存对应的代价
        history.append(best_score)  # 记录截至当前的最低代价
    return best, history  # 返回动作与搜索过程而不是模型参数
plan, search_history = cem(start)  # 从共同起点搜索一次开环计划
fig, ax = plt.subplots(figsize=(6, 3))  # 创建搜索代价图
ax.plot(range(1, 6), search_history, marker="o")  # 显示每轮保留的最佳候选代价
ax.set(xlabel="CEM iteration", ylabel="Best cost so far")  # 说明横轴不是神经网络训练轮数
plt.show()  # 显示搜索逐步聚焦的结果


### CEM 曲线具体表示什么

横轴是同一次决策内部的五轮搜索。每一轮都重新提出动作块、预测后果并评分；曲线下降表示找到了代价更低的具体候选，水平段表示这一轮没有刷新最佳结果。

代价同时含终点距离、沿途距离和动作幅度，因此纵轴不是米，也不是任务成功率。代码保留的是一条实际评估过的最佳动作块，而不是盲目执行所有较好候选的逐项平均值。


## 3. 执行两步后重新观察，会改变什么

两种方式从相同起点出发，并在第 6 步结束后受到相同外部位移。开环方式一次搜索十二步并全部执行；重新规划方式每执行两步就读取实际位置，再从这个位置搜索新的十二步计划，但只执行新计划的前两步。

扰动不会提前告诉规划器。请把左侧的路径跳变与右侧第 6 步的竖虚线对应起来，再观察两种方式之后怎样接近目标。


In [ ]:
disturbance = np.array([-0.22, 0.20])  # 定义两种方式共同经历的外部位移
open_states = [start.copy()]  # 保存开环执行轨迹
closed_states = [start.copy()]  # 保存滚动执行轨迹
buffer = None  # 保存滚动控制当前准备执行的两步动作
for step in range(12):  # 在相同长度内比较两种执行方式
    open_next = open_states[-1] + plan[step]  # 开环始终采用最初搜索到的动作
    if step % 2 == 0:  # 每执行两步便重新观察并规划
        buffer, _ = cem(closed_states[-1])  # 从最新真实位置重新搜索
    closed_next = closed_states[-1] + buffer[step % 2]  # 只执行新计划的前两步
    if step == 5:  # 在第六步结束时施加外部扰动
        open_next = open_next + disturbance  # 扰动开环系统的实际位置
        closed_next = closed_next + disturbance  # 对滚动系统施加同样的扰动
    open_states.append(open_next)  # 保存实际开环位置而不是模型想象
    closed_states.append(closed_next)  # 保存实际滚动控制位置
open_states = np.asarray(open_states)  # 整理开环轨迹
closed_states = np.asarray(closed_states)  # 整理闭环轨迹
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))  # 用路径与距离两幅图解释反馈
axes[0].plot(open_states[:, 0], open_states[:, 1], "o--", label="Open loop", markersize=3)  # 画出开环实际路径
axes[0].plot(closed_states[:, 0], closed_states[:, 1], "o-", label="Replan every 2 steps", markersize=3)  # 画出滚动实际路径
axes[0].scatter(*goal, marker="*", s=140, color="black", label="Goal")  # 标明目标位置
axes[0].set(xlabel="x", ylabel="y", aspect="equal")  # 保持横纵比例一致
axes[0].legend(fontsize=8)  # 显示路径含义
open_error = np.linalg.norm(open_states - goal, axis=1)  # 计算实际开环位置与目标的距离
closed_error = np.linalg.norm(closed_states - goal, axis=1)  # 计算实际滚动位置与目标的距离
axes[1].plot(open_error, "--", label="Open loop")  # 显示开环距离随时间变化
axes[1].plot(closed_error, label="Replanning")  # 显示滚动距离随时间变化
axes[1].axvline(6, color="gray", linestyle=":", label="Disturbance")  # 标出扰动发生的时刻
axes[1].set(xlabel="Step", ylabel="Distance to goal")  # 标明这里是物理距离而非规划总代价
axes[1].legend(fontsize=8)  # 标明曲线与事件
fig.tight_layout()  # 避免图例和标签挤在一起
plt.show()  # 展示同一扰动之后两种执行方式的分歧
print(f"最终距离：开环 {open_error[-1]:.3f}，滚动 {closed_error[-1]:.3f}")  # 报告实际计算出的最终误差
assert np.all(np.abs(plan) <= limit)  # 验证执行的开环计划满足本例动作范围
assert np.isfinite(closed_states).all()  # 验证滚动轨迹没有无效数值


### 最后两幅图要按时间顺序阅读

左图中的星号是目标。第 6 步附近的突然位移来自外部扰动，不是 CEM 主动选择的大动作。开环方式仍执行扰动前制定的剩余动作，因此后续命令服务于旧起点；重新规划方式在下一个两步边界读到真实位置，再从新起点搜索。

右图显示每个真实执行时刻到目标的距离，竖虚线与左图的扰动是同一事件。若重新规划最终距离较小，只能说明这个教学环境中的反馈有效。它没有碰撞、模型学习误差和推理延迟，而且重新规划调用了更多次搜索，不能据此宣称所有真实机器人任务都更安全或计算更便宜。
